In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
ollama_api_key = 'ollama'

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:8]}")
else:
    print("DeepSeek API Key not set")

In [ ]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

ollama_url = "http://localhost:11434/v1"
anthropic_url = "https://api.anthropic.com/v1/"
deepseek_url = "https://api.deepseek.com"
# gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
# gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
MARKDOWN_PROMPT = "Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."

In [ ]:
SYSTEM_PROMPT = (
    "You are a basic assistant. "
    "You solve simple tasks. "
    "Your answers are serious and straight to the point. "
    + MARKDOWN_PROMPT
)

In [ ]:
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_claude(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
      ]
    stream = anthropic.chat.completions.create(
        model='claude-sonnet-4-5-20250929',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_deepseek(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
      ]
    stream = deepseek.chat.completions.create(
        model='deepseek-reasoner',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_llama(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model='llama3.2',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_gpt_oss(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model='gpt-oss:20b',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_model(prompt, model):
    if model=="LLama":
        result = stream_llama(prompt)
    if model=="Gpt OSS":
        result = stream_gpt_oss(prompt)
    elif model=="DeepSeek":
        result = stream_deepseek(prompt)
    elif model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["LLama", "Gpt OSS", "GPT", "Claude", "DeepSeek"], label="Select model", value="Ollama")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Tell me a dark fact about the world. Keep it brief.", "Ollama"],
            ["Tell me a dark fact about the world. Keep it brief.", "Gpt OSS"],
            ["Tell me a dark fact about the world. Keep it brief.", "DeepSeek"],
            ["Tell me a dark fact about the world. Keep it brief.", "GPT"],
            ["Tell me a dark fact about the world. Keep it brief.", "Claude"],
        ], 
    flagging_mode="never"
    )
view.launch(inbrowser=True)